## Multinomial Logistic Regression

 Multinomial Logistic Regression via Gradient Descent

$$
\mathbf{z} = W^T\mathbf{x}
$$

$$
\hat{y}_k
=
\frac{e^{z_k}}
{\sum_{j=1}^{K}e^{z_j}}
$$

$$
L_B(W)
=
-\frac{1}{|B|}
\sum_{(\mathbf{x}_i,\mathbf{y}_i)\in B}
\sum_{k=1}^{K}
y_{ik}\log \hat{y}_{ik}
$$


In [1]:
import math

# Dataset
# Classes: Apple / Orange / Pear / Grape
# The first element 1 is the bias term.
x = [
    [1.0,  0.5,  1.0],
    [1.0,  0.4,  0.8],
    [1.0,  0.6,  1.2],
    [1.0, -1.0, -1.0],
    [1.0, -0.8, -1.2],
    [1.0, -1.2, -0.8],
    [1.0,  1.5,  1.8],
    [1.0,  1.8,  1.4],
    [1.0,  1.2,  1.6],
    [1.0, -1.0,  0.5],
    [1.0, -1.2,  0.8],
    [1.0, -0.8,  0.3],
]

y = [
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0, 0.0],
    [1.0, 0.0, 0.0, 0.0],
    [1.0, 0.0, 0.0, 0.0],
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 1.0],
    [0.0, 0.0, 0.0, 1.0],
    [0.0, 0.0, 0.0, 1.0],
]

# Hyperparameters
learning_rate = 0.2
num_epochs = 100
batch_size = 1

# Initial parameters
W = [
    [ 0.2,  0.1,  0.5, -0.4],
    [ 0.4,  0.2,  1.0,  0.2],
    [ 0.4,  0.1,  1.0,  0.0],
]

num_samples = len(x)
num_features = len(x[0])
num_classes = len(y[0])

for epoch in range(num_epochs):
    for start_idx in range(0, num_samples, batch_size):

        x_batch = x[start_idx:start_idx + batch_size]
        y_batch = y[start_idx:start_idx + batch_size]

        # 1) Forward
        y_hat = []

        for i in range(len(x_batch)):
            z = []

            for k in range(num_classes):
                score = 0.0

                for j in range(num_features):
                    score = score + W[j][k] * x_batch[i][j]

                z.append(score)

            exp_z = [math.exp(z_k - max(z)) for z_k in z]
            exp_sum = sum(exp_z)

            y_hat.append([value / exp_sum for value in exp_z])

        # 2) Loss
        loss = 0.0

        for i in range(len(x_batch)):
            for k in range(num_classes):
                loss = loss - y_batch[i][k] * math.log(y_hat[i][k])

        loss = loss / len(x_batch)

        ##########################
        # 3) Compute gradients for the batch
        # grad_W[j][k] = dL/dW[j][k]
        # = 1/B * sum((y_hat[k] - y[k]) * x[j])
        
        grad_W = [[0.0 for k in range(num_classes)] for j in range(num_features)]
        for j in range(num_features):
            for k in range(num_classes):
                grad_W[j][k] = grad_W[j][k]+ ((y_hat[i][k] - y_batch[i][k]) * x_batch[i][j])
                grad_W[j][k] = grad_W[j][k] / len(x_batch)

        ##########################

        # 4) Update
        for j in range(num_features):
            for k in range(num_classes):
                W[j][k] = W[j][k] - learning_rate * grad_W[j][k]

print("W =")
for row in W:
    print(row)


W =
[1.043293383525178, 2.562872962135141, -3.5105285111683546, 0.30436216550804096]
[-1.6880742777713482, 1.2547828463389976, 6.615008627014038, -4.381717195581692]
[-4.4234090886381345, 1.1910407625913066, 2.2320959329591794, 2.500272393087654]


## PyTorch

This time, we implement Gradient Descent using PyTorch.

`nn.Linear(2, 4)` represents the following model.
$$
\mathbf{z} = W^T\mathbf{x}
$$


In [4]:
import torch
import torch.nn as nn

# Dataset
x = torch.tensor([
    [ 0.5,  1.0],
    [ 0.4,  0.8],
    [ 0.6,  1.2],
    [-1.0, -1.0],
    [-0.8, -1.2],
    [-1.2, -0.8],
    [ 1.5,  1.8],
    [ 1.8,  1.4],
    [ 1.2,  1.6],
    [-1.0,  0.5],
    [-1.2,  0.8],
    [-0.8,  0.3],
])
y = torch.tensor([
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0, 0.0],
    [1.0, 0.0, 0.0, 0.0],
    [1.0, 0.0, 0.0, 0.0],
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 1.0],
    [0.0, 0.0, 0.0, 1.0],
    [0.0, 0.0, 0.0, 1.0],
])

# Hyperparameters
learning_rate = 0.2
num_epochs = 100
batch_size = 1

num_samples = len(x)

# Model: z = W^T*x
model = nn.Linear(2, 4)

# Initial parameters
W = torch.tensor([
    [ 0.2,  0.1,  0.5, -0.4],
    [ 0.4,  0.2,  1.0,  0.2],
    [ 0.4,  0.1,  1.0,  0.0],
])

with torch.no_grad():
    model.bias.copy_(W[0])
    model.weight.copy_(W[1:].T)

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)  # Gradient Descent

for epoch in range(num_epochs):
    for start_idx in range(0, num_samples, batch_size):

        x_batch = x[start_idx:start_idx + batch_size]
        y_batch = y[start_idx:start_idx + batch_size]

        # 1) Forward
        z = model(x_batch)
        y_hat = torch.softmax(z, dim=1)

        # 2) Loss
        # L = -1/B * sum(y * log(y_hat))
        loss = -torch.mean(torch.sum(y_batch * torch.log(y_hat), dim=1))

        # 3) Gradient
        optimizer.zero_grad()
        loss.backward()

        # 4) Update
        optimizer.step()

W = torch.cat([
    model.bias.reshape(1, -1),
    model.weight.T
], dim=0)

print("W =")
print(W)
# 최종 Loss 계산
with torch.no_grad():
    z = model(x)
    y_hat = torch.softmax(z, dim=1)
    final_loss = -torch.mean(
        torch.sum(y * torch.log(y_hat), dim=1)
    )

print("Final Loss =", final_loss.item())

W =
tensor([[ 1.0433,  2.5629, -3.5105,  0.3044],
        [-1.6881,  1.2548,  6.6150, -4.3817],
        [-4.4234,  1.1910,  2.2321,  2.5003]], grad_fn=<CatBackward0>)
Final Loss = 0.059583839029073715
